In [174]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
max_iters = 1000
block_size = 8
batch_size = 4
#eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250
#dropout = 0.2

cuda


In [185]:
with open('TestDataSet_DonQuijote.txt', 'r', encoding='Utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '#', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '¡', '«', '»', '¿', 'Á', 'É', 'Í', 'Ñ', 'Ó', 'Ú', 'à', 'á', 'é', 'í', 'ï', 'ñ', 'ó', 'ù', 'ú', 'ü', '—', '\ufeff']


In [176]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
#print(data[:100])

In [177]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs: ')
#print(x.shape)
print(x)
print('targets: ')
print(y)

inputs: 
tensor([[70, 91,  0, 54, 66, 64, 66,  1],
        [56, 69, 69, 66, 58, 52, 71, 60],
        [ 1, 63, 66,  1, 70, 60, 56, 65],
        [72, 56,  1, 70, 56,  1, 52, 63]], device='cuda:0')
targets: 
tensor([[91,  0, 54, 66, 64, 66,  1, 70],
        [69, 69, 66, 58, 52, 71, 60, 73],
        [63, 66,  1, 70, 60, 56, 65, 55],
        [56,  1, 70, 56,  1, 52, 63, 56]], device='cuda:0')


In [178]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y =get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [179]:

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('Cuando la entrada es: ',context,' el objetivo: ', target)

Cuando la entrada es:  tensor([99])  el objetivo:  tensor(43)
Cuando la entrada es:  tensor([99, 43])  el objetivo:  tensor(59)
Cuando la entrada es:  tensor([99, 43, 59])  el objetivo:  tensor(56)
Cuando la entrada es:  tensor([99, 43, 59, 56])  el objetivo:  tensor(1)
Cuando la entrada es:  tensor([99, 43, 59, 56,  1])  el objetivo:  tensor(39)
Cuando la entrada es:  tensor([99, 43, 59, 56,  1, 39])  el objetivo:  tensor(69)
Cuando la entrada es:  tensor([99, 43, 59, 56,  1, 39, 69])  el objetivo:  tensor(66)
Cuando la entrada es:  tensor([99, 43, 59, 56,  1, 39, 69, 66])  el objetivo:  tensor(61)


In [180]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



éRF0àt:M"íBV?¡!t9E0ly—e,C;BeíBÑi)Á?(u[ày3hÑGYBR;]4í6é;3
r-A2« ARÍ1*
Z*5ÁIcP;gñ«40'Lù0[íl0]wÍíá W(AjGQTS«B6lXÁ9AóóOVpx2JÓz«n¿]aà-,ÁÍHíN#R0'—hG"w]KOC?Shy.oZJGQwHKbs5 AefUA—*Kü4ZoóñíjJuÓ"ÁùHSNCaF]LbrE.ü2IQpnY
:ÓoxWbásZ[Á—TJG5Mó—EDÑf﻿?.H-p7vIù!4ít-TgrF﻿Uá?¡BC3x,A*,oDvcàbTeúF93¿94Q*:F¡ï;dd(Í¿áïzíSvúDuÓFO 1OZ6CúÉ*BQkhc-JkászCQ!m2n2üà-.?4yAP;[m,wZg2JÓFobZbXg'Hu'*phá¿]àVvb2Xà,Naáj5jÑiFxùR)Td)6"¿o1Gf»k»»vrkwH?)OñY5?;Jzx¿bp
ÍIsoóü)3C¡(;1G,«ü3iV»"4a54V54sóK—qG]?Y¡YAP3fé]wáÓl"rDcà﻿G4hhSZ﻿'ï56Y9Y﻿Ñov,G6!IskR


In [181]:
#optimizador de pytorch
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    #muestra de un monton de datos
    xb, yb = get_batch('train')
    
    #evaluar la pérdida
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.990, val loss: 4.988
step: 250, train loss: 4.924, val loss: 4.925
step: 500, train loss: 4.842, val loss: 4.858
step: 750, train loss: 4.800, val loss: 4.789
4.406491279602051


In [182]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


39Y 9EdX—y0¡vrvy"M"¿üÍkqYK#SúEeñió;wà1[7xN#ñL1V»Q»É¡as5uÍfMu;3d A-9OíYj);aAóceúhY#v(ÚX[]4YïïtDfp(ÍüP;QtíP»mu.?.OK—WFSupÑJ¡BKÓzbhwÓïE,ÉGÁs—11á
NZRIAeaAp¿iíI6GV?óyà Tc*i#xo*FnlÁÍVu"ÉuX!tJdX!,dVA*0áf—1I;go'?'L],CA*hl1yQóP:««fÓù4(ó Fx«cÑBKàÚrvv?Sá[JN4ÍHvÚ6EqéwvBq?.#jóxU(N]ÑofD#:MMñ9Y6Ó"V-ïOÁ93áxiàijuÉf﻿NF﻿R —í!uz﻿é]Z—Ñ5
LN#«ÓúqJGOSYA*5xV?T»V?SOn TX»7n2icY6
ÚK#qxmQN;611á 9üDnV1a úà]àxútkJ3[n!.-X!'
;AóyZb-áEj6ÚL?]'6L)ù.pgH7íNw)KyL?
¿ü5-Ó
WüpO(9áÍic9;P6SóU:.I'Úr0C9BJ4VàNïum«àyït:GBe*m32I.!P ¿wLxÉówbéká
